___
# <center>Distribuições de probabilidade</center>
___

## Aula 08

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * reconhecer qual distribuição descreve um fenômeno jurídico a partir do enunciado;
 * calcular probabilidades com a binomial e com a Poisson no scipy;
 * distinguir contagem (discreta) de tempo de espera (contínua);
 * ler um histograma da base como uma distribuição observada.

Curto, como o de terça. A escolha da distribuição foi feita na lousa; aqui você
confere as contas e vê o formato de cada uma.


___
<div id="indice"></div>

## Índice

- [O que muda de ontem para hoje](#ideia)

- [Contar sucessos: a binomial](#binomial)

- [Contar chegadas: a Poisson](#poisson)

- [Esperar: a exponencial](#exponencial)

- [A distribuição que a base mostra](#base)

- [RESUMO](#resumo)


___
<div id="ideia"></div>

# O que muda de ontem para hoje

Na terça calculamos a probabilidade de **um** evento por vez. Uma distribuição
responde de uma vez só para **todos** os resultados possíveis: qual a chance de
0, de 1, de 2, de 3, e assim por diante.

O `scipy.stats` tem uma função por distribuição, e todas se usam do mesmo jeito.


In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from scipy import stats

pd.set_option("display.max_columns", 30)


✔️ **O padrão do scipy**, igual para todas as distribuições:

| método | o que devolve |
|---|---|
| `.pmf(k)` | chance de sair **exatamente** k (só nas discretas) |
| `.pdf(x)` | altura da curva em x (só nas contínuas) |
| `.cdf(k)` | chance de sair **k ou menos** |
| `.rvs(n)` | sorteia n valores |


[Volta ao Índice](#indice)


___
<div id="binomial"></div>

# Contar sucessos: a binomial

**O escritório vai interpor 10 recursos. Cada um tem 30% de chance de ser
provido, e um não interfere no outro. Quantos serão providos?**

Número fixo de tentativas, mesma chance em cada uma, independentes: binomial.


In [ ]:
n, p = 10, 0.30

# A chance de exatamente 3 serem providos
stats.binom.pmf(3, n, p)


In [ ]:
# A distribuição inteira, de 0 a 10
quantos = np.arange(0, n + 1)

dist = pd.DataFrame({
    "providos": quantos,
    "chance": stats.binom.pmf(quantos, n, p),
})

dist.round(4)


In [ ]:
(
    ggplot(dist)
    + aes(x="providos", y="chance")
    + geom_col(fill="#3ACC9F")
    + scale_x_continuous(breaks=quantos)
    + labs(x="recursos providos em 10", y="probabilidade",
           title="Binomial com n = 10 e p = 0,30")
    + theme_minimal()
)


O pico está em 3, que é $10 \times 0{,}30$. Mas repare no resto: sair 1 ou sair
5 é perfeitamente possível. **Um resultado longe da média não é prova de que a
chance estava errada.**


**✍️ Agora você.** A pergunta do enunciado era "pelo menos 4". Como `.cdf(3)` dá a chance de 3 ou menos, a chance de 4 ou mais é o complementar.


In [ ]:
1 - stats.binom.cdf(________, n, p)


[Volta ao Índice](#indice)


___
<div id="poisson"></div>

# Contar chegadas: a Poisson

**Uma vara recebe em média 7 processos novos por dia útil. Qual a chance de
chegarem mais de 15 num dia?**

Aqui não existem "10 tentativas": poderiam chegar 40. Contagem num intervalo de
tempo, sem teto natural, é Poisson. O único parâmetro é a média, $\lambda$.


In [ ]:
lam = 7

# Mais de 15 é o complementar de "15 ou menos"
1 - stats.poisson.cdf(15, lam)


In [ ]:
chegadas = np.arange(0, 21)

poisson = pd.DataFrame({
    "processos": chegadas,
    "chance": stats.poisson.pmf(chegadas, lam),
})

(
    ggplot(poisson)
    + aes(x="processos", y="chance")
    + geom_col(fill="#730D9F")
    + labs(x="processos que chegam num dia", y="probabilidade",
           title="Poisson com média 7")
    + theme_minimal()
)


Menos de 1% dos dias teriam mais de 15 chegadas. Isso é o que permite
dimensionar equipe: não pelo dia médio, e sim pelo dia ruim.


[Volta ao Índice](#indice)


___
<div id="exponencial"></div>

# Esperar: a exponencial

**Quanto tempo entre a distribuição do recurso e o julgamento?**

Tempo não é contagem: pode ser 2,4 anos. É uma variável contínua, sempre
positiva, com muitos casos rápidos e uma minoria muito lenta. Esse formato é o
da exponencial.

Aqui vamos usar dados de verdade, e comparar o modelo com eles.


In [ ]:
# A base de câmaras, a mesma do Projeto 2. A coluna `tempo` traz os anos
# entre a distribuição do recurso e o julgamento.
CAMARAS = "https://jtrecenti.github.io/cdad2-202662/_shared/dados/camaras.csv"

camaras = pd.read_csv(CAMARAS)
tempo = camaras["tempo"].dropna()

tempo.describe().round(2)


In [ ]:
media = tempo.mean()

# A chance de o recurso passar de 5 anos, pelo modelo exponencial
1 - stats.expon.cdf(5, scale=media)


E na base de verdade, quantos passaram de 5 anos?


In [ ]:
(tempo > 5).mean()


Os dois números ficam próximos, mas não iguais. **A exponencial é um modelo do
formato, não uma cópia da base**: nenhum recurso é julgado no dia seguinte, e a
exponencial acha que isso seria o mais comum de todos.


In [ ]:
grade = np.linspace(0, tempo.max(), 300)

curva = pd.DataFrame({
    "anos": grade,
    "densidade": stats.expon.pdf(grade, scale=media),
})

(
    ggplot()
    + geom_histogram(aes(x="tempo", y="after_stat(density)"),
                     data=camaras.dropna(subset=["tempo"]),
                     bins=40, fill="#C9CCD2", color="white")
    + geom_line(aes(x="anos", y="densidade"), data=curva,
                color="#E50505", size=1.2)
    + labs(x="anos entre a distribuição e o julgamento", y="densidade",
           title="O que a base tem, e o modelo por cima")
    + theme_minimal()
)


⚠️ **A altura da curva não é probabilidade.** Numa contínua, a chance de dar
*exatamente* 365,0 dias é zero: o que tem probabilidade é um intervalo, e ela é
a área embaixo da curva. Por isso a contínua usa `.pdf` e não `.pmf`, e por
isso a pergunta é sempre "mais que", "menos que" ou "entre".


[Volta ao Índice](#indice)


___
<div id="base"></div>

# A distribuição que a base mostra

Até aqui foram distribuições teóricas. A base também tem distribuições, e elas
são o histograma da aula 6.


In [ ]:
URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")
penas = criminal.dropna(subset=["pena_anos"]).query("pena_anos <= 30")

(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=25, fill="#F89D49", color="white")
    + labs(x="pena em anos", y="acórdãos",
           title="A distribuição observada das penas")
    + theme_minimal()
)


<div id="ex1"></div>

### EXERCÍCIO 1

Olhe o histograma e responda em uma frase cada:

1. essa distribuição parece mais com qual das que vimos hoje: a simétrica em
   torno da média, ou a de cauda longa para a direita?
2. a média é maior ou menor que o valor mais comum? Confira com
   `penas["pena_anos"].mean()` e `.mode()`.
3. por que usar a média da pena como "a pena típica" pode enganar num relatório?


In [ ]:
print("média :", round(penas["pena_anos"].mean(), 2))
print("mediana:", round(penas["pena_anos"].median(), 2))
print("moda   :", penas["pena_anos"].mode().iloc[0])


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

| a pergunta do caso | distribuição | no scipy |
|---|---|---|
| deu certo ou não, uma vez só | Bernoulli | `stats.bernoulli` |
| quantos deram certo em n tentativas | binomial | `stats.binom` |
| quantos aconteceram no período | Poisson | `stats.poisson` |
| quantas tentativas até o primeiro | geométrica | `stats.geom` |
| quanto tempo até acontecer | exponencial | `stats.expon` |
| medida que se acumula em torno de um centro | normal | `stats.norm` |

**A frase para levar:** a distribuição não vem dos dados, vem do enunciado. Quem
escolhe é a pergunta: contagem com teto, contagem sem teto, ou tempo de espera.


[Volta ao Índice](#indice)
